# DemandWise — Entendimento dos dados

Este notebook realiza a inspeção inicial das bases da competição **Store Item Demand Forecasting Challenge**. O objetivo é validar estrutura, tipos, completude, cobertura temporal e características básicas da variável `sales` antes da análise exploratória.

## 1. Configuração e carregamento

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "data" / "raw" / "train.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Execute o notebook a partir da raiz do projeto ou da pasta notebooks/.")

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.csv"

print(f"Raiz do projeto: {PROJECT_ROOT.resolve()}")

In [ ]:
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test = pd.read_csv(TEST_PATH, parse_dates=["date"])

print("Bases carregadas com sucesso.")

## 2. Primeiras linhas

In [ ]:
display(train.head())
display(test.head())

## 3. Dimensões e tipos de dados

In [ ]:
dimensions = pd.DataFrame(
    {
        "base": ["train", "test"],
        "linhas": [train.shape[0], test.shape[0]],
        "colunas": [train.shape[1], test.shape[1]],
    }
)
display(dimensions)

In [ ]:
print("Tipos — treino")
display(train.dtypes.to_frame("dtype"))
print("Tipos — teste")
display(test.dtypes.to_frame("dtype"))

## 4. Valores ausentes

In [ ]:
missing = pd.concat(
    [
        train.isna().sum().rename("train_ausentes"),
        test.isna().sum().rename("test_ausentes"),
    ],
    axis=1,
).fillna(0).astype(int)
display(missing)

## 5. Cobertura temporal e cardinalidade

In [ ]:
coverage = pd.DataFrame(
    {
        "base": ["train", "test"],
        "data_inicial": [train["date"].min(), test["date"].min()],
        "data_final": [train["date"].max(), test["date"].max()],
        "lojas": [train["store"].nunique(), test["store"].nunique()],
        "produtos": [train["item"].nunique(), test["item"].nunique()],
    }
)
display(coverage)

## 6. Distribuição básica de vendas

In [ ]:
display(train["sales"].describe().to_frame().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train["sales"], bins=50, ax=axes[0], color="#2563EB")
axes[0].set(title="Distribuição das vendas diárias", xlabel="Vendas", ylabel="Frequência")
sns.boxplot(x=train["sales"], ax=axes[1], color="#93C5FD")
axes[1].set(title="Dispersão das vendas", xlabel="Vendas")
plt.tight_layout()
plt.show()

## 7. Primeiras observações

In [ ]:
observations = [
    f"O treino possui {len(train):,} registros entre {train['date'].min():%d/%m/%Y} e {train['date'].max():%d/%m/%Y}.",
    f"O teste possui {len(test):,} registros entre {test['date'].min():%d/%m/%Y} e {test['date'].max():%d/%m/%Y}.",
    f"A base de treino reúne {train['store'].nunique()} lojas e {train['item'].nunique()} produtos.",
    f"Foram encontrados {int(train.isna().sum().sum())} valores ausentes no treino e {int(test.isna().sum().sum())} no teste.",
    f"As vendas variam de {train['sales'].min()} a {train['sales'].max()}, com média de {train['sales'].mean():.2f} unidades por registro.",
    "A estrutura é um painel temporal: cada série deve ser analisada por combinação de loja e produto.",
    "A validação dos futuros modelos deverá respeitar a ordem do tempo, sem divisão aleatória.",
]

for number, observation in enumerate(observations, start=1):
    print(f"{number}. {observation}")

### Conclusão

Esta inspeção estabelece a qualidade e a cobertura da matéria-prima do projeto. A próxima etapa será investigar tendência, sazonalidade e diferenças de demanda entre lojas e produtos no notebook `02_exploratory_analysis.ipynb`.